[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/708san/CV_data/blob/codex/library-ocr-runtime/notebooks/colab/generation_smoke_test.ipynb)

# SDXL・LoRAのColab動作確認

このNotebookはライブラリの画像生成機能を試すための確認用です。Google Colabで **T4 GPU** を選んでください。SDXL本体とLoRAを初回に取得するため、数GBの空き容量と時間が必要です。

利用するモデルは4件とも配布元から直接取得します。重みを取得する前に次のセルでモデルのライセンス表示・学習元データ・注意点を確認してください。画像生成の出力に正確な文字や事実を期待しないでください。

使用例はプロンプトの例です。自分の目的に合わせて、実行前に文章を自由に書き換えられます。Image-to-Imageには合成画像か、利用者がColabへ明示アップロードした画像を選べます。Colabへのアップロードは利用者自身が行い、ライブラリは画像を別サービスへ自動送信しません。個人情報や共有する権利のない画像は使わないでください。
ランタイムの種類でT4 GPUを選んでから「すべてのセルを実行」してください。依存関係の導入後に再起動を求められた場合は、再起動して残りのセルを実行します。

In [ ]:
%pip install 'cv-data-lesson[diffusion] @ git+https://github.com/708san/CV_data.git@635fa30db261ae5b3d43ab87c9e5f9e66a78bd1a'

In [ ]:
import torch
from IPython.display import display
from cvlesson import loras

if not torch.cuda.is_available():
    raise RuntimeError('GPUが有効ではありません。ランタイムのタイプを変更し、GPUを選択してください。')

gpu_name = torch.cuda.get_device_name(0)
print(f'GPU: {gpu_name}')
if 'T4' not in gpu_name:
    print('この環境はT4ではありません。実行結果には実際のGPU名を記録してください。')

models = loras.list_models()
for model in models:
    print(f"\n{model['id']} — {model['name_ja']}")
    print(f"  モデルのライセンス表示: {model['license']['name']}")
    print(f"  学習元: {model['training_dataset']['repository']}")
    print(f"  学習元の条件: {model['training_dataset']['license']}")
    print(f"  利用上の説明: {model['rights_note']}")
    print(f"  モデルの特性: {model['capabilities_ja']}")
    print(f"  使用例（一例）: {'、'.join(model['use_examples_ja'])}")
    print(f"  注意: {model['limitations_ja']}")

print('上の権利表示と注意を確認してから、次のセルを実行してください。')

## 4件のLoRAを順に試す

各LoRAを1件ずつ読み込み、画像を1枚ずつ生成します。画像は `/content/cvlesson_lora_smoke_results/` に保存されます。失敗したモデルがあっても残りを試し、最後に成功・失敗を表示します。

以下のプロンプトは動作確認用の例です。別の題材に書き換えて試せます。

In [ ]:
import gc
from pathlib import Path
from PIL import Image, ImageDraw
from cvlesson import generation

result_dir = Path('/content/cvlesson_lora_smoke_results')
result_dir.mkdir(parents=True, exist_ok=True)

# Falseなら合成画像、TrueならColabへ自分の画像をアップロードします。
USE_OWN_IMAGE = False
if USE_OWN_IMAGE:
    from google.colab import files

    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('画像ファイルを選んでください')
    with Image.open(next(iter(uploaded))) as opened:
        source_image = opened.convert('RGB')
else:
    source_image = Image.new('RGB', (512, 512), '#d9efff')
    draw = ImageDraw.Draw(source_image)
    draw.rectangle((0, 330, 512, 512), fill='#76a96b')
    draw.ellipse((55, 55, 150, 150), fill='#ffd45c')
    draw.polygon([(260, 170), (150, 360), (390, 360)], fill='#4c7c58')
    draw.rectangle((300, 280, 340, 390), fill='#80624b')
display(source_image)

cases = [
    ('fluent_emoji_3d', 'text-to-image', '明るい色の太陽をモチーフにした、丸みのある3Dアイコン'),
    ('watercolor_sdxl', 'image-to-image', '丘と木々のある風景をやわらかな水彩画風に描く'),
    ('pixel_art_sdxl', 'text-to-image', '赤いリンゴの単体ピクセルアート素材'),
    ('thomas_cole_landscape', 'image-to-image', '丘と川の風景をHudson River School風の油彩画にする'),
]

results = []
for model_id, task, prompt in cases:
    generator = None
    try:
        print(f'\n読み込み中: {model_id} ({task})')
        generator = generation.load(task=task, lora=model_id, device='cuda')
        kwargs = {'num_inference_steps': 6, 'guidance_scale': 5.0, 'seed': 42, 'width': 512, 'height': 512}
        if task == 'image-to-image':
            kwargs.update(image=source_image, strength=0.65)
        image = generator.generate(prompt, **kwargs)
        output_path = result_dir / f'{model_id}.png'
        image.save(output_path)
        results.append((model_id, '成功', str(output_path)))
        display(image)
        print(f'保存先: {output_path} / 画像サイズ: {image.size}')
    except Exception as exc:
        results.append((model_id, '失敗', str(exc)))
        print(f'{model_id} の実行に失敗しました: {type(exc).__name__}: {exc}')
    finally:
        if generator is not None:
            del generator
        gc.collect()
        torch.cuda.empty_cache()

print('\n--- 結果 ---')
for model_id, status, detail in results:
    print(f'{model_id}: {status} — {detail}')
print(f'\nGPU最大メモリ使用量: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB')
print(f'生成画像の保存先: {result_dir}')
failures = [row for row in results if row[1] != '成功']
if failures:
    raise RuntimeError(f'{len(failures)}件のLoRA実行に失敗しました。上のエラーとGPU環境を確認してください。')
print('4件すべての画像生成に成功しました。')

## 結果を確認する

- 4つのモデルで画像が1枚ずつ生成されたか確認します。失敗したモデルがあれば、エラーとGPU名を記録します。
- それぞれの絵柄や、文章・入力画像との対応を見ます。目的に応じてプロンプトを変え、結果がどう変化するか試せます。
- 出力は試作品です。生成画像だけで現地の状態や安全性を判断せず、文字は別のアプリで正しく入れてください。
- このNotebookを実行した結果はColab上の実環境テストです。リポジトリへ生成画像やモデル重みをアップロードしないでください。